# Gold Layer — Star Schema

## Objective

The Gold layer models the cleansed Silver data into a **Star Schema** optimized for
Business Intelligence consumption, per Apex Retail's Medallion Architecture.

### Star Schema Design

| Table | Type | Contents |
|---|---|---|
| dim_customer | Dimension | Customer attributes, SCD Type 2 history (effective_start_date, effective_end_date, is_current) |
| dim_product | Dimension | Product catalogue details, SCD Type 1 (current state only) |
| dim_promotion | Dimension | Promotion types and identifiers |
| dim_date | Dimension | Freshly generated calendar attributes spanning the transaction date range |
| fact_sales | Fact (Central) | Transaction-level metrics, linked to all dimensions via surrogate keys |

### Grain

`fact_sales` is at the **one row per transaction_id** grain.

### Registration

All five tables are registered as managed Delta tables in Unity Catalog under the
`workspace.GOLD_tables` schema, making them queryable via standard SQL by downstream
BI consumers.

This notebook also computes five KPIs (Section: Business Reporting) directly on the
star schema using PySpark, per the "no external dashboarding tools" constraint.

In [0]:
# ============================================================
# Imports
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, current_timestamp, current_date,
    row_number, monotonically_increasing_id,
    year, month, dayofmonth, dayofweek, quarter, date_format,
    sum as _sum, avg, count, round as _round, expr, sequence, explode
)
from pyspark.sql.window import Window

# ============================================================
# Base Paths (same as Silver notebook)
# ============================================================

BASE_PATH = "/Volumes/workspace/default/apex_retail_data"

BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH   = f"{BASE_PATH}/gold"

# ============================================================
# Silver Paths (read from — already built)
# ============================================================

CUSTOMER_SILVER_PATH = f"{SILVER_PATH}/customer"
PRODUCT_SILVER_PATH  = f"{SILVER_PATH}/product"
SALES_SILVER_PATH    = f"{SILVER_PATH}/sales"

# ============================================================
# Gold Paths (write to — building now)
# ============================================================

DIM_CUSTOMER_PATH  = f"{GOLD_PATH}/dim_customer"
DIM_PRODUCT_PATH   = f"{GOLD_PATH}/dim_product"
DIM_PROMOTION_PATH = f"{GOLD_PATH}/dim_promotion"
DIM_DATE_PATH      = f"{GOLD_PATH}/dim_date"
FACT_SALES_PATH    = f"{GOLD_PATH}/fact_sales"

GOLD_SCHEMA = "GOLD_tables"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
print(f"Schema '{GOLD_SCHEMA}' ready.")

Schema 'GOLD_tables' ready.


In [0]:
# ============================================================
# Helper Functions (same as Silver notebook)
# ============================================================

def read_delta(path: str):
    return spark.read.format("delta").load(path)

def write_delta(df, path: str, mode: str):
    (
        df.write
        .format("delta")
        .mode(mode)
        .save(path)
    )
    print(f"Successfully written to:\n{path}")

In [0]:
# ============================================================
# dim_customer
# ============================================================

customer_silver = read_delta(CUSTOMER_SILVER_PATH)

dim_customer = customer_silver.select(
    "customer_sk",
    "customer_id",
    "age", "gender", "income_bracket", "loyalty_program",
    "membership_years", "churned", "marital_status",
    "number_of_children", "education_level", "occupation",
    "customer_zip_code", "customer_city", "customer_state",
    "effective_start_date", "effective_end_date", "is_current"
)

print(f"dim_customer Rows : {dim_customer.count()}")

write_delta(df=dim_customer, path=DIM_CUSTOMER_PATH, mode="overwrite")

dim_customer Rows : 1059
Successfully written to:
/Volumes/workspace/default/apex_retail_data/gold/dim_customer


In [0]:
# ============================================================
# dim_product
# ============================================================

product_silver = read_delta(PRODUCT_SILVER_PATH)

dim_product = product_silver.select(
    "product_sk",
    "product_id",
    "product_name", "product_brand", "product_category",
    "product_rating", "product_review_count", "product_stock",
    "product_return_rate", "product_size", "product_weight",
    "product_color", "product_material",
    "product_manufacture_date", "product_expiry_date",
    "product_shelf_life", "unit_price"
)

print(f"dim_product Rows : {dim_product.count()}")

write_delta(df=dim_product, path=DIM_PRODUCT_PATH, mode="overwrite")

dim_product Rows : 1041
Successfully written to:
/Volumes/workspace/default/apex_retail_data/gold/dim_product


In [0]:
# ============================================================
# dim_promotion
# ============================================================

sales_silver = read_delta(SALES_SILVER_PATH)

dim_promotion = (
    sales_silver
    .select("promotion_id", "promotion_type")
    .distinct()
    .withColumn(
        "promotion_sk",
        row_number().over(Window.orderBy("promotion_id"))
    )
    .select("promotion_sk", "promotion_id", "promotion_type")
)

print(f"dim_promotion Rows : {dim_promotion.count()}")

write_delta(df=dim_promotion, path=DIM_PROMOTION_PATH, mode="overwrite")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_promotion Rows : 877
Successfully written to:
/Volumes/workspace/default/apex_retail_data/gold/dim_promotion


In [0]:
# ============================================================
# dim_date
# ============================================================

date_bounds = sales_silver.selectExpr(
    "min(transaction_date) as min_date",
    "max(transaction_date) as max_date"
).collect()[0]

dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{date_bounds['min_date']}'),
            to_date('{date_bounds['max_date']}'),
            interval 1 day
        )) as full_date
    """)
    .withColumn("date_sk", row_number().over(Window.orderBy("full_date")))
    .withColumn("year", year("full_date"))
    .withColumn("month", month("full_date"))
    .withColumn("day", dayofmonth("full_date"))
    .withColumn("quarter", quarter("full_date"))
    .withColumn("day_name", date_format("full_date", "EEEE"))
    .withColumn("month_name", date_format("full_date", "MMMM"))
    .withColumn(
        "is_weekend",
        when(dayofweek("full_date").isin([1, 7]), True).otherwise(False)
    )
    .select("date_sk", "full_date", "year", "month", "day",
            "quarter", "day_name", "month_name", "is_weekend")
)

print(f"dim_date Rows : {dim_date.count()}")

write_delta(df=dim_date, path=DIM_DATE_PATH, mode="overwrite")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_date Rows : 1460
Successfully written to:
/Volumes/workspace/default/apex_retail_data/gold/dim_date


In [0]:
# ============================================================
# fact_sales
# ============================================================

fact_sales = (
    sales_silver.alias("s")

    .join(
        dim_customer.filter(col("is_current") == True).alias("c"),
        col("s.customer_id") == col("c.customer_id"),
        "left"
    )

    .join(
        dim_product.alias("p"),
        col("s.product_id") == col("p.product_id"),
        "left"
    )

    .join(
        dim_promotion.alias("promo"),
        col("s.promotion_id") == col("promo.promotion_id"),
        "left"
    )

    .join(
        dim_date.alias("d"),
        col("s.transaction_date") == col("d.full_date"),
        "left"
    )

    .select(
        col("s.sales_sk"),
        col("s.transaction_id"),
        col("c.customer_sk"),
        col("p.product_sk"),
        col("promo.promotion_sk"),
        col("d.date_sk"),
        col("s.quantity"),
        col("s.unit_price"),
        col("s.discount_applied"),
        col("s.total_sales"),
        col("s.payment_method"),
        col("s.store_location"),
        col("s.transaction_hour"),
        col("s.holiday_season"),
        col("s.season"),
        col("s.weekend")
    )
)

print(f"fact_sales Rows : {fact_sales.count()}")

# sanity check — unmatched FK joins would show as nulls
fact_sales.select(
    _sum(when(col("customer_sk").isNull(), 1).otherwise(0)).alias("missing_customer_sk"),
    _sum(when(col("product_sk").isNull(), 1).otherwise(0)).alias("missing_product_sk"),
    _sum(when(col("date_sk").isNull(), 1).otherwise(0)).alias("missing_date_sk")
).show()

write_delta(df=fact_sales, path=FACT_SALES_PATH, mode="overwrite")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


fact_sales Rows : 1649
+-------------------+------------------+---------------+
|missing_customer_sk|missing_product_sk|missing_date_sk|
+-------------------+------------------+---------------+
|                  0|               753|             44|
+-------------------+------------------+---------------+

Successfully written to:
/Volumes/workspace/default/apex_retail_data/gold/fact_sales


# Business Reporting (KPI Generation)

## Objective

Using the Gold Star Schema built above, this section computes five Key Performance
Indicators entirely within PySpark DataFrame operations and Spark SQL, satisfying
the constraint that no external BI/dashboarding tool (Power BI, Tableau, etc.) is
used — all outputs are rendered inline in this Databricks notebook.

### KPIs Computed

1. **Net Margin by Region** — gross revenue minus discounts, grouped by store region
2. **Average Order Value (AOV) by Promotion** — which promotion types drive the highest average cart values
3. **Demographic Churn Heatmap** — churn rate split by state and loyalty program membership
4. **Product Quality Index** — which product categories have the highest return rates
5. **Store Traffic by Hour** — busiest transaction hours/days for store foot traffic

Each KPI is followed by its result and a short business interpretation.

In [0]:
# ============================================================
# Unity Catalog Registration (Managed Tables)
# ============================================================

CATALOG = "workspace"
GOLD_SCHEMA = "GOLD_tables"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
print(f"Schema '{CATALOG}.{GOLD_SCHEMA}' ready.")

gold_tables = {
    "dim_customer":  dim_customer,
    "dim_product":   dim_product,
    "dim_promotion": dim_promotion,
    "dim_date":      dim_date,
    "fact_sales":    fact_sales
}

for table_name, df in gold_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.{table_name}")
    )
    print(f"Registered: {CATALOG}.{GOLD_SCHEMA}.{table_name}")

display(spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}"))

Schema 'workspace.GOLD_tables' ready.
Registered: workspace.GOLD_tables.dim_customer
Registered: workspace.GOLD_tables.dim_product


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Registered: workspace.GOLD_tables.dim_promotion
Registered: workspace.GOLD_tables.dim_date
Registered: workspace.GOLD_tables.fact_sales


database,tableName,isTemporary
gold_tables,dim_customer,false
gold_tables,dim_date,false
gold_tables,dim_product,false
gold_tables,dim_promotion,false
gold_tables,fact_sales,false


In [0]:
# ============================================================
# KPI 1: Net Margin by Region
# ============================================================

kpi_net_margin_by_region = (
    fact_sales.alias("f")
    .join(dim_customer.alias("c"), "customer_sk")
    .groupBy(col("c.customer_state").alias("region"))
    .agg(
        _round(_sum(col("f.total_sales")), 2).alias("gross_revenue"),
        _round(_sum(col("f.total_sales") * col("f.discount_applied")), 2).alias("total_discount"),
        _round(_sum(col("f.total_sales")) - _sum(col("f.total_sales") * col("f.discount_applied")), 2).alias("net_margin")
    )
    .orderBy(col("net_margin").desc())
)

display(kpi_net_margin_by_region)

In [0]:
# ============================================================
# KPI 2: Average Order Value (AOV) by Promotion Type
# ============================================================

kpi_aov_by_promotion = (
    fact_sales.alias("f")
    .join(dim_promotion.alias("p"), "promotion_sk")
    .groupBy(col("p.promotion_type"))
    .agg(_round(avg(col("f.total_sales")), 2).alias("avg_order_value"))
    .orderBy(col("avg_order_value").desc())
)

display(kpi_aov_by_promotion)

In [0]:
# ============================================================
# KPI 3: Demographic Churn Heatmap (State x Loyalty Program)
# ============================================================

kpi_churn_heatmap = (
    dim_customer
    .filter(col("is_current") == True)
    .groupBy("customer_state", "loyalty_program")
    .agg(
        count("*").alias("total_customers"),
        _sum(when(col("churned") == "Yes", 1).otherwise(0)).alias("churned_customers")
    )
    .withColumn(
        "churn_rate_pct",
        _round((col("churned_customers") / col("total_customers")) * 100, 2)
    )
    .orderBy(col("churn_rate_pct").desc())
)

display(kpi_churn_heatmap)

In [0]:
# ============================================================
# KPI 4: Product Quality Index (Return Rate by Category)
# ============================================================

kpi_product_quality = (
    dim_product
    .groupBy("product_category")
    .agg(_round(avg("product_return_rate"), 4).alias("avg_return_rate"))
    .orderBy(col("avg_return_rate").desc())
)

display(kpi_product_quality)

In [0]:
# ============================================================
# KPI 5: Store Traffic by Hour & Day of Week
# ============================================================

kpi_store_traffic = (
    fact_sales
    .groupBy("transaction_hour")
    .agg(count("*").alias("transaction_count"))
    .orderBy(col("transaction_count").desc())
)

display(kpi_store_traffic)

## KPI Interpretation

**1. Net Margin by Region**
Region-level net margin (gross revenue minus discounts) highlights which store
regions are the strongest revenue contributors after promotional spend. Regions
with high gross revenue but a large gap to net margin indicate heavy discounting —
a candidate for promotion strategy review.

**2. Average Order Value (AOV) by Promotion Type**
Comparing AOV across promotion types shows which promotion mechanics (e.g. BOGO,
percentage-off, flat discount) drive customers to spend more per basket versus
which simply move volume at lower ticket sizes. Promotions with high AOV are
stronger candidates to scale.

**3. Demographic Churn Heatmap**
Cross-tabulating churn rate by state and loyalty program membership surfaces
whether the loyalty program is actually reducing churn, and whether churn is
geographically concentrated — both useful for targeting retention campaigns.

**4. Product Quality Index**
Categories with the highest average return rate flag potential quality control
or listing-accuracy issues (e.g. sizing, description mismatch) worth investigating
with the sourcing/QA team before they affect brand trust at scale.

**5. Store Traffic by Hour**
Peak transaction hours identify optimal staffing windows and the best time slots
for flash promotions or app notifications to capture in-the-moment demand.